# Ontario caribou core habitat, 2015–2025 — v4

**What changed from v3.** v3 used ECCC's 2020 footprint for roads and infrastructure. ECCC maps roads
visible in Landsat at 1:50,000, about 18% of the MNRF road network in these ranges, so v3 missed most
forest-access roads and overstated core habitat (2020: −2.5% overall, −9.3% in Nipigon once roads are added).
v4 adds the Ontario sources named in Mackey et al. (2024), with roads entering the annual series by construction year.

| Component | Source | Years | Buffer |
|---|---|---|---|
| Harvest | ARI 2025 ∪ national harvest 1985–2022 (ARI year wins) | 1976 → year | 500 m |
| Roads | MNRF Road Segments (by construction year) ∪ National Road Network | built ≤ year | 500 m |
| Other human footprint | ECCC 2020 anthropogenic 500 m footprint ∪ OLCC v2 classes 25, 27, 28 ∪ FRI unclassified | static | 500 m |
| Fire | ARI fire ∪ National Fire Database ∪ ECCC 40-year fire | rolling 40 years | none |
| Other natural | ARI blowdown, drought, unknown | rolling 40 years | none |

**Road years.** MNRF records a construction year with a qualifier. *Actual* and *approximate* roads enter in
that year. *Before* roads (built before the stated year, often much earlier) and roads without a usable year
are treated as present in every analysis year. NRN, OLCC, FRI unclassified and ECCC are single snapshots applied
to all years.

Outputs go to `outputs_v4/`; `outputs_v3/` is untouched. Set `CARIBOU_RANGES=Sydney` to test one range.

In [1]:
import os
import time
import warnings
from pathlib import Path

import cv2
import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import rasterio
from rasterio.features import rasterize
from rasterio.vrt import WarpedVRT
from rasterio.windows import from_bounds
from scipy import ndimage
from shapely.geometry import box

warnings.filterwarnings('ignore')
MARKER = Path('data') / 'Caribou_range_boundary' / 'Caribou_range_boundary.shp'
ROOT = next(c for c in [Path.cwd(), *Path.cwd().parents] if (c / MARKER).exists())
DATA, RAW = ROOT / 'data', ROOT / 'data' / 'raw'
PI = RAW / 'paper_inputs'
V3 = ROOT / 'outputs_v3'
OUT = ROOT / 'outputs_v4'
for sub in ['01_potential_habitat', '02_disturbance', '03_undisturbed_habitat', '04_statistics', '05_mspa']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

ARI_GDB = DATA / 'ARI_AnalysisReadyInventory2025' / 'ARI_AnalysisReadyInventory2025' / 'ARI_wFU.gdb'
ARI_EVENTS = DATA / 'interim' / 'ari_events_gdb'
NATIONAL_HARVEST = RAW / 'national_harvest' / 'CA_Forest_Harvest_1985-2022' / 'CA_Forest_Harvest_1985-2022.tif'
LANDCOVER = '/vsizip/' + (RAW / 'landcover_2019' / 'CA_forest_VLCE2_2019.zip').as_posix() + '/CA_forest_VLCE2_2019.tif'
MNRF_ROADS = PI / 'mnrf_roads' / 'Non_Sensitive.gdb'
NRN_GPKG = next((PI / 'nrn_ontario').rglob('NRN_ON_*_GPKG_en.gpkg'))
NRN_LAYER = next(n for n, _ in pyogrio.list_layers(NRN_GPKG) if n.endswith('ROADSEG'))
NFDB = [PI / 'nfdb_fire' / 'NFDB_poly_1972to2020_20250630.shp', PI / 'nfdb_fire' / 'NFDB_poly_2021to2024_20250630.shp']
OLCC = next((PI / 'olcc_v2').rglob('OLCC_V2_TIFF.tif'))
ECCC_DIR = RAW / 'disturbance_2020'

YEARS = list(range(2015, 2026))
HARVEST_START, WINDOW = 1976, 40
BUFFER_M, RES_M, PIXEL_HA, NODATA = 500, 30, 0.09, 255
OLCC_CLASSES = [25, 27, 28]
OTHER_DEPTYPES = ['BLOWDOWN', 'DROUGHT', 'UNKNOWN']
PAPER_RANGES = ['Berens', 'Brightsand', 'Churchill', 'Kesagami', 'Nipigon', 'Pagwachuan', 'Sydney']
RUN = [r.strip() for r in os.environ.get('CARIBOU_RANGES', ','.join(PAPER_RANGES)).split(',') if r.strip()]
PAPER = pd.DataFrame([('Berens', 1612106, 46.4), ('Brightsand', 1525297, 65.6), ('Churchill', 2035815, 49.0),
                      ('Kesagami', 3373204, 53.5), ('Nipigon', 2928933, 74.4), ('Pagwachuan', 2165773, 67.8),
                      ('Sydney', 578902, 50.0)], columns=['RANGE_NAME', 'paper_range_ha', 'paper_pct_disturbed'])
TYPES = ['harvest', 'roads', 'human_other', 'fire', 'natural_other']   # priority order for allocating overlaps
for p in [ARI_GDB, ARI_EVENTS, NATIONAL_HARVEST, MNRF_ROADS, NRN_GPKG, OLCC, *NFDB]:
    assert p.exists(), p
print('ranges to run:', RUN)

ranges to run: ['Berens', 'Brightsand', 'Churchill', 'Kesagami', 'Nipigon', 'Pagwachuan', 'Sydney']


## 1. Ranges and the study-area boundary (range ∩ Area of Undertaking)

In [2]:
ranges = gpd.read_file(DATA / 'Caribou_range_boundary' / 'Caribou_range_boundary.shp').to_crs(3978)
ranges = ranges[ranges.RANGE_NAME.isin(PAPER_RANGES)].set_index('RANGE_NAME')
bnd = gpd.read_file(ARI_GDB, layer='_FMU_FN_PARK_Boundary_simp2m').to_crs(3978)
bnd['geometry'] = bnd.geometry.make_valid()
STUDY_AREA = bnd.loc[bnd.TYPE == 'MU'].geometry.union_all()
AVAILABLE = {n for n, _ in pyogrio.list_layers(ARI_GDB)}
ECCC_TOTAL = {p.stem.split('_')[-1]: p for p in ECCC_DIR.glob('eccc_2020_total_disturbance_*.geojson')}
print('study area %.1f M ha' % (STUDY_AREA.area / 1e10))

study area 44.1 M ha


## 2. Helpers (core extraction identical to v3)

In [3]:
def burn(geoms, profile, touched=False):
    shapes = [(g, 1) for g in geoms if g is not None and not g.is_empty]
    if not shapes:
        return np.zeros((profile['height'], profile['width']), bool)
    return rasterize(shapes, out_shape=(profile['height'], profile['width']), transform=profile['transform'],
                     fill=0, dtype='uint8', all_touched=touched).astype(bool)

def burn_years(geoms, years, profile):
    shapes = [(g, int(y)) for g, y in zip(geoms, years) if g is not None and not g.is_empty and 1800 < y < 2100]
    if not shapes:
        return np.zeros((profile['height'], profile['width']), 'int16')
    return rasterize(shapes, out_shape=(profile['height'], profile['width']), transform=profile['transform'],
                     fill=0, dtype='int16')

def buffer_metres(mask, metres=BUFFER_M):
    if not mask.any():
        return mask
    dist = cv2.distanceTransform((~mask).astype('uint8'), cv2.DIST_L2, cv2.DIST_MASK_PRECISE)
    return dist * RES_M <= metres

def read_bbox(path, frame, layer=None, columns=None, where=None):
    info = pyogrio.read_info(path, layer=layer)
    crs = info['crs'] or 'EPSG:4326'
    bb = gpd.GeoSeries([frame], crs=3978).to_crs(crs).total_bounds
    df = pyogrio.read_dataframe(path, layer=layer, columns=columns, bbox=tuple(bb), where=where)
    return (df.set_crs(crs) if df.crs is None else df).to_crs(3978)

def write_raster(path, array, profile):
    prof = profile.copy()
    prof.update(driver='GTiff', count=1, dtype='uint8', nodata=NODATA, compress='LZW')
    with rasterio.open(path, 'w', **prof) as dst:
        dst.write(array.astype('uint8'), 1)

def core_size_classes(undisturbed_bool, valid_mask):
    structure = np.ones((3, 3), dtype=bool)
    core = ndimage.binary_erosion(undisturbed_bool, structure=structure, border_value=0)
    labels, count = ndimage.label(core, structure=structure)
    areas = np.bincount(labels.ravel()) * PIXEL_HA
    bins = np.array([0, .25, .5, 1, 2, 3, 4, 5, 10, 25, 50, 100, 250, 500, 1000, 10000, 50000, 250000, 500000, np.inf])
    classes = np.digitize(areas, bins, right=False) - 1
    out = classes[labels].astype('uint8') + 1
    out[~core] = 0
    out[~valid_mask] = NODATA
    return out, int(count), (core & valid_mask)

def risk(p):
    return pd.cut(p, bins=[-np.inf, 10, 35, 45, 75, np.inf], labels=['Very Low', 'Low', 'Moderate', 'High', 'Very High'])

## 3. Annual disturbance, undisturbed habitat, core habitat and statistics

In [4]:
results, road_log = [], []
t0 = time.time()
for name in RUN:
    key = name.lower()
    rg = ranges.loc[name].geometry
    src_pot = next((V3 / '01_potential_habitat').glob(key + '_*.tif'))
    with rasterio.open(src_pot) as s:
        potential = s.read(1)
        profile = s.profile.copy()
    write_raster(OUT / '01_potential_habitat' / src_pot.name, potential, profile)
    valid = potential != NODATA
    habitat = (potential == 1) & valid
    bounds = rasterio.transform.array_bounds(profile['height'], profile['width'], profile['transform'])
    frame = box(*bounds)
    study = burn([STUDY_AREA.intersection(rg)], profile) & valid
    with rasterio.open(LANDCOVER) as s:
        lc = s.read(1, window=from_bounds(*bounds, transform=s.transform).round_offsets().round_lengths())
    land = valid & (lc != 20)

    # harvest year (ARI wins, national fills), inventory fire and other natural depletion years
    ev = gpd.read_file(ARI_EVENTS / (key + '_ari_events.gpkg'))
    ev['YRDEP'] = pd.to_numeric(ev.YRDEP, errors='coerce')
    ari_h = burn_years(ev.loc[ev.DEPTYPE == 'HARVEST'].geometry, ev.loc[ev.DEPTYPE == 'HARVEST'].YRDEP, profile)
    fire_ari = burn_years(ev.loc[ev.DEPTYPE == 'FIRE'].geometry, ev.loc[ev.DEPTYPE == 'FIRE'].YRDEP, profile)
    other_year = burn_years(ev.loc[ev.DEPTYPE.isin(OTHER_DEPTYPES)].geometry, ev.loc[ev.DEPTYPE.isin(OTHER_DEPTYPES)].YRDEP, profile)
    with rasterio.open(NATIONAL_HARVEST) as s:
        nat = s.read(1, window=from_bounds(*bounds, transform=s.transform).round_offsets().round_lengths()).astype('int16')
    assert nat.shape == valid.shape
    harvest_year = np.where(ari_h > 0, ari_h, nat)
    del ari_h, nat, ev

    # static human footprint: ECCC anthropogenic 500 m, OLCC 25/27/28 and FRI unclassified (both buffered)
    eccc_total = burn(gpd.read_file(ECCC_TOTAL[name]).set_crs(3978, allow_override=True).geometry.buffer(0), profile)
    eccc_anthro = burn(gpd.read_file(ECCC_DIR / f'eccc_2020_anthro500m_{name}.geojson').set_crs(3978, allow_override=True).geometry.buffer(0), profile)
    eccc_fire = eccc_total & ~eccc_anthro
    with rasterio.open(OLCC) as s, WarpedVRT(s, crs=profile['crs'], transform=profile['transform'], width=profile['width'],
                                             height=profile['height'], resampling=rasterio.enums.Resampling.nearest) as v:
        olcc = np.isin(v.read(1), OLCC_CLASSES) & valid
    ucl = []
    for lyr in [n for n in bnd[bnd.intersects(rg)].INV_NAME.dropna().unique() if n in AVAILABLE]:
        g = read_bbox(ARI_GDB, frame, layer=lyr, columns=['POLYTYPE'], where="POLYTYPE = 'UCL'")
        if len(g):
            ucl.append(g)
    ucl_mask = burn(pd.concat(ucl).geometry if ucl else [], profile)
    human_other = (eccc_anthro | buffer_metres(olcc) | buffer_metres(ucl_mask)) & valid
    del olcc, ucl_mask, eccc_total, eccc_anthro

    # roads: MNRF by construction year + NRN
    mnrf = read_bbox(MNRF_ROADS, frame, layer='MNRF_ROAD_SEGMENT', columns=['YEAR_CONSTRUCTED', 'YEAR_CONSTRUCTED_MODIFIER'])
    yc = pd.to_numeric(mnrf.YEAR_CONSTRUCTED, errors='coerce')
    dated = mnrf.YEAR_CONSTRUCTED_MODIFIER.isin(['Actual', 'Approximate']) & yc.between(1900, 2100)
    mnrf['entry'] = np.where(dated, yc, 0)             # 0 = present in every analysis year
    nrn = read_bbox(NRN_GPKG, frame, layer=NRN_LAYER, columns=[])
    roads_static = burn(pd.concat([mnrf.loc[mnrf.entry <= YEARS[0]].geometry, nrn.geometry]), profile, touched=True)
    road_log.append(dict(range=name, mnrf_segments=len(mnrf), undated_or_before=int((mnrf.entry == 0).sum()),
                         built_2016_2025=int(mnrf.entry.between(YEARS[0] + 1, YEARS[-1]).sum()), nrn_segments=len(nrn)))

    fire_polys = pd.concat([read_bbox(p, frame, columns=['YEAR']) for p in NFDB])
    fire_polys['YEAR'] = pd.to_numeric(fire_polys.YEAR, errors='coerce')

    roads_now = roads_static.copy()
    for year in YEARS:
        if year > YEARS[0]:
            new = mnrf.loc[mnrf.entry == year]
            if len(new):
                roads_now |= burn(new.geometry, profile, touched=True)
        layers = {
            'harvest': buffer_metres((harvest_year >= HARVEST_START) & (harvest_year <= year)) & valid,
            'roads': buffer_metres(roads_now) & valid,
            'human_other': human_other,
            'fire': (((fire_ari >= year - WINDOW) & (fire_ari <= year))
                     | burn(fire_polys.loc[fire_polys.YEAR.between(year - WINDOW, year)].geometry.buffer(0), profile)
                     | eccc_fire) & valid,
            'natural_other': (other_year >= year - WINDOW) & (other_year <= year) & valid,
        }
        disturbance = np.zeros(valid.shape, bool)
        for k in TYPES:
            disturbance |= layers[k]
        undisturbed = habitat & ~disturbance
        core_classes, patch_count, core = core_size_classes(undisturbed, valid)
        write_raster(OUT / '02_disturbance' / f'{key}_total_disturbance_{year}.tif', np.where(valid, disturbance, NODATA), profile)
        write_raster(OUT / '03_undisturbed_habitat' / f'{key}_undisturbed_habitat_{year}.tif', np.where(valid, undisturbed, NODATA), profile)
        write_raster(OUT / '05_mspa' / f'{key}_core_patch_size_class_{year}.tif', core_classes, profile)

        row = {'RANGE_NAME': name, 'YEAR': year}
        for zone, mask in [('', valid), ('_study', study), ('_land', land), ('_study_land', study & land)]:
            ha = lambda m: float((m & mask).sum()) * PIXEL_HA
            row[f'area{zone}_ha'] = ha(mask)
            row[f'disturbed_area{zone}_ha'] = ha(disturbance)
            row[f'potential_habitat{zone}_ha'] = ha(habitat)
            row[f'undisturbed_habitat{zone}_ha'] = ha(undisturbed)
            row[f'core_habitat{zone}_ha'] = ha(core)
            assigned = np.zeros(valid.shape, bool)
            for k in TYPES:
                row[f'{k}_alone{zone}_ha'] = ha(layers[k])
                row[f'{k}_assigned{zone}_ha'] = ha(layers[k] & ~assigned)
                assigned |= layers[k]
        row['core_patch_count'] = patch_count
        results.append(row)
        del layers, disturbance, undisturbed, core_classes, core
    print(f'[{time.time() - t0:6.0f}s] {name}: 2015-2025 done', flush=True)
    del harvest_year, fire_ari, other_year, human_other, roads_now, roads_static, eccc_fire, potential, valid, habitat, study, land, lc

stats = pd.DataFrame(results)
for zone in ['', '_study', '_land', '_study_land']:
    stats[f'disturbance_percent{zone}'] = 100 * stats[f'disturbed_area{zone}_ha'] / stats[f'area{zone}_ha']
    stats[f'risk_class{zone}'] = risk(stats[f'disturbance_percent{zone}'])
# v3-compatible names used by the story page and the app scripts
stats['range_area_ha'] = stats['area_ha']
stats['study_area_ha'] = stats['area_study_ha']
stats['harvest_buffered_ha'] = stats['harvest_alone_ha']
stats['fire_ha'] = stats['fire_alone_ha']
suffix = '' if set(RUN) == set(PAPER_RANGES) else '_' + '_'.join(r.lower() for r in RUN)
stats.to_csv(OUT / '04_statistics' / f'annual_caribou_habitat_statistics_2015_2025{suffix}.csv', index=False)
pd.DataFrame(road_log).to_csv(OUT / '04_statistics' / f'road_inputs{suffix}.csv', index=False)
print('statistics written', len(stats), 'rows')

[   148s] Berens: 2015-2025 done


[   290s] Brightsand: 2015-2025 done


[   447s] Churchill: 2015-2025 done


[   761s] Kesagami: 2015-2025 done


[   986s] Nipigon: 2015-2025 done


[  1254s] Pagwachuan: 2015-2025 done


[  1309s] Sydney: 2015-2025 done


statistics written 77 rows


## 4. Comparison with Mackey et al. (2024), 2020
The paper reports disturbance for range ∩ study area. Our land-only basis leaves out water (VLCE2 class 20); five ranges matched the paper on that basis in the rebuild test.

In [5]:
y20 = stats[stats.YEAR == 2020].merge(PAPER, on='RANGE_NAME')
cmp_ = y20[['RANGE_NAME', 'paper_range_ha', 'study_area_ha', 'paper_pct_disturbed',
            'disturbance_percent_study', 'disturbance_percent_study_land', 'disturbance_percent']].copy()
cmp_['area_ratio'] = cmp_.study_area_ha / cmp_.paper_range_ha
cmp_['gap_study'] = cmp_.disturbance_percent_study - cmp_.paper_pct_disturbed
cmp_['gap_study_land'] = cmp_.disturbance_percent_study_land - cmp_.paper_pct_disturbed
cmp_.to_csv(OUT / '04_statistics' / f'comparison_with_paper_2020{suffix}.csv', index=False)
print('mean absolute gap: study %.1f | study land-only %.1f points' % (cmp_.gap_study.abs().mean(), cmp_.gap_study_land.abs().mean()))
cmp_.round(1)

mean absolute gap: study 8.7 | study land-only 9.2 points


,RANGE_NAME,paper_range_ha,study_area_ha,paper_pct_disturbed,disturbance_percent_study,disturbance_percent_study_land,disturbance_percent,area_ratio,gap_study,gap_study_land
0,Berens,1612106,1582194.1,46.4,40.8,43.7,47.7,1.0,-5.6,-2.7
1,Brightsand,1525297,1513391.0,65.6,64.3,68.9,50.5,1.0,-1.3,3.3
2,Churchill,2035815,2010000.0,49.0,45.8,51.1,44.6,1.0,-3.2,2.1
3,Kesagami,3373204,3360253.5,53.5,58.6,60.1,45.4,1.0,5.1,6.6
4,Nipigon,2928933,2897135.3,74.4,49.4,51.1,40.2,1.0,-25.0,-23.3
5,Pagwachuan,2165773,2150925.0,67.8,68.7,68.9,36.1,1.0,0.9,1.1
6,Sydney,578902,571530.2,50.0,69.8,75.4,70.3,1.0,19.8,25.4


## 5. v3 → v4 change

In [6]:
v3 = pd.read_csv(V3 / '04_statistics' / 'annual_caribou_habitat_statistics_2015_2025.csv')
ch = stats[['RANGE_NAME', 'YEAR', 'disturbance_percent', 'core_habitat_ha']].merge(
    v3[['RANGE_NAME', 'YEAR', 'disturbance_percent', 'core_habitat_ha']], on=['RANGE_NAME', 'YEAR'], suffixes=('_v4', '_v3'))
ch['disturbance_change_pts'] = ch.disturbance_percent_v4 - ch.disturbance_percent_v3
ch['core_change_pct'] = 100 * (ch.core_habitat_ha_v4 / ch.core_habitat_ha_v3 - 1)
ch.to_csv(OUT / '04_statistics' / f'v3_to_v4_change{suffix}.csv', index=False)
ch[ch.YEAR.isin([2015, 2020, 2025])].round(1)

,RANGE_NAME,YEAR,disturbance_percent_v4,core_habitat_ha_v4,disturbance_percent_v3,core_habitat_ha_v3,disturbance_change_pts,core_change_pct
0,Berens,2015,48.7,203665.0,46.7,208569.2,2.0,-2.4
5,Berens,2020,47.7,206184.7,46.1,209792.2,1.6,-1.7
10,Berens,2025,55.2,174392.4,52.7,176998.9,2.5,-1.5
11,Brightsand,2015,51.3,275343.0,48.3,279617.6,3.1,-1.5
16,Brightsand,2020,50.5,279026.2,47.9,282533.1,2.6,-1.2
21,Brightsand,2025,47.8,264578.8,46.2,270238.1,1.5,-2.1
22,Churchill,2015,44.7,249485.0,41.7,256014.1,3.0,-2.6
27,Churchill,2020,44.6,250191.6,41.7,256147.9,2.8,-2.3
32,Churchill,2025,46.7,232377.5,43.6,241336.5,3.2,-3.7
33,Kesagami,2015,45.3,1206101.8,43.2,1239887.4,2.2,-2.7
